# Lab 5: Neural Networks (LSTM and GRU) for JPM

**Owner:** Member 5 · **Marking items covered:** 4. Modelling quality (Advanced items 1 and 2), 2. Honest validation

**The question this notebook answers:** Do LSTM or GRU networks trained on windows of our features beat the naive forecast, and are their results stable across seeds?

> **How to use this starter (delete this box when you finish).** Steps 1 and 2 already work: run them
> first. Every other step says what to do and which course notebook to copy from. Replace each `# TODO`
> cell with your code, and each *italic* placeholder with your own words about **JPM's** results.
> Your full instructions are in your section of the team guide.

## Step 1: Set up

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tslab import config
from tslab.data import load_market_data
from tslab.team import build_features_jpm, make_dataset_jpm, FEATURE_GROUPS_JPM, ALL_FEATURES_JPM   # OUR features
from tslab.validation import walk_forward_splits, walk_forward_predict, describe_splits
from tslab.metrics import (compare_models, diebold_mariano, dm_verdict, directional_accuracy,
                           directional_accuracy_test, out_of_sample_r2, rmse, mae,
                           simple_backtest, backtest_summary)
from tslab.plots import plot_walk_forward_layout, plot_price_forecasts
from sklearn.preprocessing import StandardScaler
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras
from keras import layers
from tslab.nnutils import (set_seed, make_windows, build_lstm, build_gru, build_cnn,
                           fit_with_early_stopping, predict_returns)
from tslab.plots import plot_learning_curves

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(config.SEED)
print("Target stock:", config.TARGET_TICKER)

## Step 2: The data and the walk-forward folds

Every notebook in our project uses these lines, so that every model is scored on the same days.

In [ ]:
frame = load_market_data()                                              # one row per JPM trading day
dataset_all = make_dataset_jpm(frame, horizon=1)                        # includes the last row (no target yet)
latest_row = dataset_all.iloc[[-1]]                                     # kept aside for a real forecast
dataset = dataset_all.dropna(subset=["target_return"])
splits = list(walk_forward_splits(len(dataset), config.INITIAL_TRAIN, config.TEST_SIZE))

print("Rows in the modelling table:", len(dataset))                     # expect 2789
print("Walk-forward folds         :", len(splits))                      # expect 29
print("First test day:", dataset.index[splits[0][1][0]].date(), "| last test day:", dataset.index[splits[-1][1][-1]].date())

## Step 3: Settings and features

Close other programs first (TensorFlow needs memory). Our feature list = Notebook 6's list plus the financial sector.

In [ ]:
WINDOW = 20
MAX_EPOCHS = 60
NN_TEST_SIZE = 252
VALIDATION_SHARE = 0.15          # validation = the LAST 15% of each training block

NN_FEATURES = [
    "ret_lag1", "ret_21d", "vol_5d", "vol_21d", "range_hl",
    "rsi_14", "macd_hist", "bb_percent_b", "dist_sma_50",
    "gap_overnight", "move_intraday", "volume_rel_20d",
    "SP500_ret_1d", "TECH_ret_1d", "GOLD_ret_1d", "BRENT_ret_1d", "USD_ret_1d",
    "VIX_level", "VIX_change_1d", "US10Y_change_5d",
    "FIN_ret_1d", "relative_to_FIN_5d",           # added by our group
]
features = dataset[NN_FEATURES].values
target = dataset["target_return"].values
print("Features per day:", len(NN_FEATURES), "| rows:", len(dataset))

## Step 4: The rules we follow

*TODO (your words): scalers fitted on training rows only; validation from the END of the training block; early stopping; a fixed seed before every model.*

## Step 5: The tempting mistake

Repeat Notebook 6, Step 3 (a network on the price) and explain in two sentences.

In [ ]:
# TODO

## Step 6: One fold with the predict-zero line

Copy Notebook 6, Step 6. Print the best validation loss next to the loss of predicting zero.

In [ ]:
# TODO

## Step 7: Tuning on validation data only

Window 10, 20, 40 x units 16, 32 x dropout 0.1, 0.3, on the first block's validation data (Notebook 6, Step 11). Show the table with the zero-loss beside it; save `outputs/nn_tuning.csv`.

In [ ]:
# TODO

## Step 8: Walk-forward test of LSTM and GRU

Copy `walk_forward_nn` from Notebook 6 and run both architectures with your chosen settings.

In [ ]:
nn_predictions = {}
# TODO

## Step 9: Three or more seeds

Seeds 42, 7, 2024 for your best network: RMSE, R² and directional accuracy per seed and for their average. Save `outputs/nn_seeds.csv`. Explain the spread.

In [ ]:
# TODO

## Step 10: Compare with naive

`compare_models` + the DM table, on the common days.

In [ ]:
# TODO

## Step 11: 63-day path from a multi-output GRU

One GRU with 63 outputs (cumulative return to days 1 to 63). Same dates as Member 4. Save `outputs/forecast_nn_path.csv`.

In [ ]:
# TODO

## Step 12: GPU (optional)

Is a GPU visible? Explain what would be needed (on Windows: WSL2) and report the CPU time.

In [ ]:
import tensorflow as tf
print("GPUs visible to TensorFlow:", tf.config.list_physical_devices("GPU"))

## Save

Other members read this file. Keep the file and column names exactly as in the team guide.

In [ ]:
if nn_predictions:
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    pd.concat(nn_predictions, axis=1).to_csv(config.OUTPUT_DIR / "predictions_nn.csv")
    print("Saved outputs/predictions_nn.csv")
else:
    print("Nothing saved yet: no networks.")

## Conclusion

*TODO (your words, about JPM's numbers). If no model beats the naive forecast, say so plainly: it is a valid finding.*